# ODI to Databricks Migration: `TRG_EMP.sql`

**Conversion Timestamp:** 2024-07-30T12:00:00Z

**Description:** This notebook converts an ODI session to load data into the `TRG_EMP` table from the `EMPLOYEES` table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "FULL", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "1001", "ETL Process Widget")
dbutils.widgets.text("ODI_SESS_NO", "0", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT ${DATASOURCE_NUM_ID} AS datasource_num_id;

CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT ${ETL_PROC_WID} AS etl_proc_wid;

CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT ${ODI_SESS_NO} AS odi_sess_no;

In [ ]:
display(spark.sql("""
SELECT
  (SELECT etl_job_type FROM v_etl_job_type) AS ETL_JOB_TYPE,
  (SELECT datasource_num_id FROM v_datasource_num_id) AS DATASOURCE_NUM_ID,
  (SELECT etl_proc_wid FROM v_etl_proc_wid) AS ETL_PROC_WID,
  (SELECT odi_sess_no FROM v_odi_sess_no) AS ODI_SESS_NO
"""))

## Load `TRG_EMP` (SCEN_TASK_NO in {30})

In [ ]:
%sql
INSERT INTO workspace.hr.trg_emp
  (
    employee_id ,
    first_name ,
    last_name ,
    email ,
    phone_number ,
    hire_date ,
    job_id ,
    salary ,
    commission_pct ,
    manager_id ,
    department_id
  )
SELECT
  employees.employee_id ,
  employees.first_name ,
  employees.last_name ,
  employees.email ,
  employees.phone_number ,
  employees.hire_date ,
  employees.job_id ,
  employees.salary ,
  employees.commission_pct ,
  employees.manager_id ,
  employees.department_id
FROM
  workspace.hr.employees AS employees;

## Validation

### Record Count in Target Table

Check the total number of records in the `trg_emp` table.

In [ ]:
%sql
SELECT COUNT(*) AS target_record_count FROM workspace.hr.trg_emp;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** All schema and table names (`HR.TRG_EMP`, `HR.EMPLOYEES`) have been converted to lowercase and prefixed with `workspace.` (e.g., `workspace.hr.trg_emp`).
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` Oracle hint has been removed as it is not applicable in Databricks Spark SQL for Delta tables.
3.  **Empty SCEN_TASK_NO:** `SCEN_TASK_NO in {10}` and `SCEN_TASK_NO in {20}` were empty and have been omitted from the generated SQL cells, as they did not contain any executable SQL statements.
4.  **Target Table DDL:** The original ODI source did not provide the DDL for `TRG_EMP`. Ensure the target table `workspace.hr.trg_emp` exists with a compatible schema (e.g., `STRING` for `VARCHAR2`, `TIMESTAMP` for `DATE` and `TIMESTAMP`, `DOUBLE` or `DECIMAL` for `NUMBER` type columns) before running this notebook.
5.  **Parameter Widgets:** Standard ETL parameter widgets have been included as per the migration guidelines, although they are not explicitly used in this specific `INSERT` statement.